# 02 · The splits — country, region, city, provider

Every slice of this dataset holds the same four metrics and the same percentiles. Only "one row per what" changes. This notebook answers the question the splits exist for:

**Where does internet quality vary — between countries, within a country, or between providers?**

| Slice | One row per… | What you can learn |
|---|---|---|
| `by_country` | country | Where the country itself sits among others (notebook 01) |
| `by_country_subdivision1` | state / province | The range *inside* one country — capital versus periphery |
| `by_country_city` | city | Which cities stand out from their own country |
| `by_country_asn` | provider | Provider differences — **compare carefully, never rank or promote** |

> **About providers (ASNs):** these data are not cleaned, and small-sample rows can mislead. There is no provider leaderboard here — instead we show how to *check one provider against its country*, with a high sample floor. That is the honest use of this slice.

## Setup — the same loader, plus one trick

Months are published asynchronously, so we use the newest month that exists in **all four slices** this notebook needs. The manifest answers that question directly.

In [ ]:
# ── Setup: imports, manifest, catalog ─────────────────────────────────────────
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

MANIFEST_URL = "https://measurementlab.net/data/stats/manifest.json"
manifest = requests.get(MANIFEST_URL, timeout=30).json()

records = []
for path, meta in manifest["files"].items():
    parts = path.split("/")
    # cache/v1/{start_ts}/{end_ts}/{slice_name}/data.parquet
    if len(parts) == 6 and parts[5] == "data.parquet":
        records.append({
            "start": pd.to_datetime(parts[2], format="%Y%m%dT%H%M%SZ"),
            "end":   pd.to_datetime(parts[3], format="%Y%m%dT%H%M%SZ"),
            "slice": parts[4],
            "url":   meta["url"],
        })

catalog = (pd.DataFrame(records)
           .sort_values(["slice", "start"])
           .reset_index(drop=True))


def month_url(slice_name, start):
    # Direct URL of one month of one slice. start: 'YYYY-MM-DD' (first of month).
    row = catalog[(catalog["slice"] == slice_name) &
                  (catalog["start"] == pd.to_datetime(start))]
    if row.empty:
        raise ValueError(f"No {slice_name} file for {start}")
    return row.iloc[0]["url"]


def latest_month(slice_name):
    # Newest month that exists for a slice, as 'YYYY-MM-DD'.
    return catalog.loc[catalog["slice"] == slice_name, "start"].max().strftime("%Y-%m-%d")

print("Catalog loaded —", len(catalog), "files,",
      catalog["slice"].nunique(), "slices.")

In [ ]:
SLICES = ["downloads_by_country", "downloads_by_country_subdivision1",
          "downloads_by_country_city", "downloads_by_country_asn"]


def latest_common_month(slice_names):
    months = None
    for s in slice_names:
        s_months = set(catalog.loc[catalog["slice"] == s, "start"])
        months = s_months if months is None else months & s_months
    return max(months).strftime("%Y-%m-%d")


MONTH = latest_common_month(SLICES)
print("Newest month present in all slices:", MONTH)

## Regions within a country

Pick a country: its states/provinces rank by your chosen metric. The *range* between the fastest and slowest region is the internal geography of quality — compare a country with a tight range (regions roughly alike) against one with a wide range (a capital far ahead of the periphery).

> **Sample counts matter most here.** A subdivision with 100 tests is a whisper, not a verdict — raise the minimum.

In [ ]:
SPLIT_METRICS = {
    "Download p50": ("download_p50", "downloads"),
    "Upload p50":   ("upload_p50",   "uploads"),
    "Latency p50":  ("latency_p50",  "downloads"),
    "Loss p50":     ("loss_p50",     "downloads"),
}
LOWER = {"Download p50": False, "Upload p50": False,
         "Latency p50": True, "Loss p50": True}

regions = pd.read_parquet(month_url("downloads_by_country_subdivision1", MONTH))
reg_countries = sorted(regions["country_code"].dropna().unique())

w_rc = widgets.Dropdown(options=reg_countries, value="US", description="Country:",
                         layout=widgets.Layout(width="200px"))
w_rm = widgets.Dropdown(options=list(SPLIT_METRICS), value="Download p50",
                         description="Metric:", layout=widgets.Layout(width="220px"))
w_rmin = widgets.IntSlider(value=500, min=0, max=50000, step=100,
                            description="Min tests:", layout=widgets.Layout(width="360px"))
w_rn = widgets.IntSlider(value=20, min=5, max=50, step=5, description="Show:",
                          layout=widgets.Layout(width="300px"))
out_r = widgets.Output()


def update_r(change=None):
    col, table = SPLIT_METRICS[w_rm.value]
    slice_name = f"{table}_by_country_subdivision1"
    data = pd.read_parquet(month_url(slice_name, MONTH))
    data = data[(data["country_code"] == w_rc.value) & (data["sample_count"] >= w_rmin.value)]
    if data.empty:
        with out_r:
            clear_output(wait=True)
            print("No regions pass the minimum test count — lower 'Min tests'.")
        return
    top = data.nsmallest(w_rn.value, col) if LOWER[w_rm.value] else data.nlargest(w_rn.value, col)
    top = top.sort_values(col, ascending=not LOWER[w_rm.value])
    with out_r:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11, max(5, len(top) * 0.36)))
        ax.barh(top["subdivision1_name"], top[col])
        ax.set_xlabel(w_rm.value)
        ax.set_title(f"Regions of {w_rc.value} by {w_rm.value} — {MONTH}")
        plt.tight_layout(); plt.show()
        print(f"Reading it aloud: {len(data)} regions of {w_rc.value} meet the sample "
              f"floor; {data['sample_count'].sum():,} tests in total.")


for w in (w_rc, w_rm, w_rmin, w_rn):
    w.observe(update_r, "value")
display(widgets.VBox([widgets.HBox([w_rc, w_rm]), w_rmin, w_rn, out_r]))
update_r()

## Cities

Same pattern, finer grain. A city with a tiny sample is noise, so the sample floor does the filtering and the chart says how many tests back each number.

In [ ]:
city = pd.read_parquet(month_url("downloads_by_country_city", MONTH))
city_countries = sorted(city["country_code"].dropna().unique())

w_cc = widgets.Dropdown(options=city_countries, value="BR", description="Country:",
                         layout=widgets.Layout(width="200px"))
w_cm = widgets.Dropdown(options=list(SPLIT_METRICS), value="Download p50",
                         description="Metric:", layout=widgets.Layout(width="220px"))
w_cmin = widgets.IntSlider(value=1000, min=0, max=50000, step=100,
                            description="Min tests:", layout=widgets.Layout(width="360px"))
w_cn = widgets.IntSlider(value=15, min=5, max=40, step=5, description="Show:",
                          layout=widgets.Layout(width="300px"))
out_c = widgets.Output()


def update_c(change=None):
    col, table = SPLIT_METRICS[w_cm.value]
    slice_name = f"{table}_by_country_city"
    data = pd.read_parquet(month_url(slice_name, MONTH))
    data = data[(data["country_code"] == w_cc.value) & (data["sample_count"] >= w_cmin.value)]
    if data.empty:
        with out_c:
            clear_output(wait=True)
            print("No cities pass the minimum test count — lower 'Min tests'.")
        return
    top = data.nsmallest(w_cn.value, col) if LOWER[w_cm.value] else data.nlargest(w_cn.value, col)
    top = top.sort_values(col, ascending=not LOWER[w_cm.value])
    with out_c:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11, max(5, len(top) * 0.36)))
        ax.barh(top["city"], top[col])
        ax.set_xlabel(w_cm.value)
        ax.set_title(f"Cities of {w_cc.value} by {w_cm.value} — {MONTH}")
        plt.tight_layout(); plt.show()
        first = top.iloc[0]
        print(f"Reading it aloud: in {w_cc.value}, {first['city']} leads at "
              f"{first[col]:.0f}, backed by {first['sample_count']:,} tests.")


for w in (w_cc, w_cm, w_cmin, w_cn):
    w.observe(update_c, "value")
display(widgets.VBox([widgets.HBox([w_cc, w_cm]), w_cmin, w_cn, out_c]))
update_c()

## Providers (ASNs) — check one provider against its country

Regulators, journalists, and consumers want provider information more than anything else in this dataset — and provider rows are exactly where un-cleaned data can **mislead**. So: no leaderboards here. Instead, one honest comparison — a single provider's median against its own country's median, from a high sample floor.

The slice even includes a human-readable provider name (`as_name`) — no lookup table needed.

In [ ]:
asn = pd.read_parquet(month_url("downloads_by_country_asn", MONTH))
asn_countries = sorted(asn["country_code"].dropna().unique())

w_ac = widgets.Dropdown(options=asn_countries, value="KE", description="Country:",
                         layout=widgets.Layout(width="200px"))
w_am = widgets.Dropdown(options=list(SPLIT_METRICS), value="Download p50",
                         description="Metric:", layout=widgets.Layout(width="220px"))
w_amin = widgets.IntSlider(value=2000, min=0, max=100000, step=100,
                            description="Min tests:", layout=widgets.Layout(width="360px"))
w_an = widgets.IntSlider(value=15, min=5, max=40, step=5, description="Show:",
                          layout=widgets.Layout(width="300px"))
out_a = widgets.Output()


def update_a(change=None):
    col, table = SPLIT_METRICS[w_am.value]
    slice_name = f"{table}_by_country_asn"
    data = pd.read_parquet(month_url(slice_name, MONTH))
    data = data[(data["country_code"] == w_ac.value) & (data["sample_count"] >= w_amin.value)]
    if data.empty:
        with out_a:
            clear_output(wait=True)
            print("No providers pass the high minimum — that is information too. "
                  "Lower 'Min tests' only with care.")
        return
    top = data.nsmallest(w_an.value, col) if LOWER[w_am.value] else data.nlargest(w_an.value, col)
    top = top.sort_values(col, ascending=not LOWER[w_am.value]).copy()
    top["label"] = top.apply(lambda r: f"{r['as_name']} (AS{r['asn']})", axis=1)
    with out_a:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11, max(5, len(top) * 0.42)))
        ax.barh(top["label"], top[col])
        ax.set_xlabel(w_am.value)
        ax.set_title(f"Providers in {w_ac.value} by {w_am.value} — {MONTH} "
                     f"(each ≥ {w_amin.value} tests)")
        plt.tight_layout(); plt.show()
        print("Reading it aloud: provider medians shown at one month, same sample floor. "
              "A comparison for your own judgement — not a ranking anyone should publish.")


for w in (w_ac, w_am, w_amin, w_an):
    w.observe(update_a, "value")
display(widgets.VBox([widgets.HBox([w_ac, w_am]), w_amin, w_an, out_a]))
update_a()

## Between or within?

One plot that puts the splits together. For one country and one metric: the **country median** is one vertical line; every region and every city sits around it as points. If the points are **tighter** than the spread you saw *across countries* in notebook 01, then the quality differences sit **between countries, not within** this one — the only way to know which story the data tell.

In [ ]:
w_bc = widgets.Dropdown(options=reg_countries, value="NG", description="Country:",
                         layout=widgets.Layout(width="200px"))
w_bm = widgets.Dropdown(options=list(SPLIT_METRICS), value="Download p50",
                         description="Metric:", layout=widgets.Layout(width="220px"))
w_bmin = widgets.IntSlider(value=1000, min=0, max=50000, step=100,
                            description="Min tests:", layout=widgets.Layout(width="360px"))
out_b = widgets.Output()


def update_b(change=None):
    col, table = SPLIT_METRICS[w_bm.value]
    dl = pd.read_parquet(month_url(f"{table}_by_country", MONTH)).set_index("country_code")
    country_val = dl.loc[w_bc.value, col]
    reg = pd.read_parquet(month_url(f"{table}_by_country_subdivision1", MONTH))
    reg = reg[(reg["country_code"] == w_bc.value) & (reg["sample_count"] >= w_bmin.value)][col]
    cty = pd.read_parquet(month_url(f"{table}_by_country_city", MONTH))
    cty = cty[(cty["country_code"] == w_bc.value) & (cty["sample_count"] >= w_bmin.value)][col]
    with out_b:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.axvline(country_val, color="black", lw=2, alpha=0.6,
                   label=f"country median {country_val:.0f}")
        for y, vals, color, label in ((1.6, reg, "steelblue", "regions"),
                                      (1.0, cty, "seagreen", "cities")):
            ax.scatter(vals, [y] * len(vals), s=18, alpha=0.5, color=color, label=label)
        ax.set_yticks([])
        ax.set_xlabel(w_bm.value)
        ax.set_title(f"{w_bc.value}: country median vs its regions and cities — {MONTH}")
        ax.legend()
        plt.tight_layout(); plt.show()
        print(f"Reading it aloud: {len(reg)} regions and {len(cty)} cities meet the "
              f"floor in {w_bc.value} — compare the internal spread with the spread "
              f"across countries you saw in notebook 01.")


for w in (w_bc, w_bm, w_bmin):
    w.observe(update_b, "value")
display(widgets.VBox([widgets.HBox([w_bc, w_bm]), w_bmin, out_b]))
update_b()

## Check yourself

**Question.** A country's cities all cluster tightly around the national median, but countries differ hugely from each other. Where does the quality difference live: within the country or between countries?

**Answer.** Between countries. The splits are the tool for answering exactly this — which is why the tutorial teaches you the "between or within" plot. Next: **03 · multiple months**, and the same data over time.